In [1]:
import pandas as pd
import numpy as np
from datasets import Dataset
from collections import Counter
import gc
import time
import psutil
import os

start_time = time.time()
process = psutil.Process(os.getpid())

def print_memory_usage(step_name):
    mem_info = process.memory_info()
    mem_gb = mem_info.rss / (1024 ** 3)

    print(f"{step_name} - Memory Usage: {mem_gb:.2f} GB")
    return mem_gb

def print_step_header(step_num, step_name):
    separator = "=" * 60

    print("\n" + separator)
    print(f"STEP {step_num}: {step_name}")
    print(separator)

initial_mem = print_memory_usage("Initial")

print(f"Start Time: {time.strftime('%H:%M:%S')}")
print(f"Initial Memory: {initial_mem:.2f} GB")

c:\Users\USER\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Initial - Memory Usage: 0.28 GB
Start Time: 07:26:10
Initial Memory: 0.28 GB


In [2]:
import torch
from torch.utils.data import Dataset, DataLoader
from datasets import load_from_disk

save_dir = "C:/Users/USER/Documents/cod3astro/ML_AI/ProteinSeq_DL/data/processed"

train_dataset_arrow = load_from_disk(f"{save_dir}/train")
val_dataset_arrow   = load_from_disk(f"{save_dir}/val")
test_dataset_arrow  = load_from_disk(f"{save_dir}/test")

print(f"Training samples:   {len(train_dataset_arrow):,}")
print(f"Validation samples: {len(val_dataset_arrow):,}")
print(f"Test samples:       {len(test_dataset_arrow):,}")

class ProteinDataset(Dataset):

    def __init__(self, hf_dataset, max_seq_length=1024):
        self.dataset = hf_dataset
        self.max_seq_length = max_seq_length

        self.label_cols = [
            col for col in hf_dataset.column_names
            if col.startswith("label_")
        ]

        self.amino_acids = "ACDEFGHIKLMNPQRSTVWY"
        self.aa_to_idx = {aa: i+1 for i, aa in enumerate(self.amino_acids)}

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        row = self.dataset[idx]

        sequence = row["sequence"]
        encoded = [self.aa_to_idx.get(aa, 0) for aa in sequence]

        encoded = encoded[:self.max_seq_length]
        if len(encoded) < self.max_seq_length:
            encoded += [0] * (self.max_seq_length - len(encoded))

        labels = [row[col] for col in self.label_cols]

        return {
            "sequence": torch.tensor(encoded, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.float32)
        }

train_dataset = ProteinDataset(train_dataset_arrow)
val_dataset   = ProteinDataset(val_dataset_arrow)
test_dataset  = ProteinDataset(test_dataset_arrow)

num_classes = len(train_dataset.label_cols)
print(f"Number of GO labels: {num_classes}")

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=8, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=8, shuffle=False)

batch = next(iter(train_loader))

print("\nSanity check:")
print("Sequence type:", type(batch["sequence"]))
print("Sequence shape:", batch["sequence"].shape)
print("Labels shape:", batch["labels"].shape)

Training samples:   71,925
Validation samples: 15,412
Test samples:       15,413
Number of GO labels: 2761

Sanity check:
Sequence type: <class 'torch.Tensor'>
Sequence shape: torch.Size([8, 1024])
Labels shape: torch.Size([8, 2761])


In [3]:
import torch.nn as nn 
import torch.optim as optim 
from sklearn.metrics import f1_score 

class SimpleProteinCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=21,
            embedding_dim=128,
            padding_idx=0
        )

        self.conv_layers = nn.Sequential(
            nn.Conv1d(128, 128, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.Conv1d(128, 64, kernel_size=5, padding=2),
            nn.ReLU(),

            nn.Conv1d(64, 32, kernel_size=7, padding=3),
            nn.ReLU()
        )

        self.global_pool = nn.AdaptiveMaxPool1d(1)

        self.classifier = nn.Sequential(
            nn.Linear(32, 128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        x = self.embedding(x)
        x = x.permute(0, 2, 1)
        x = self.conv_layers(x)
        x = self.global_pool(x)
        x = x.squeeze(-1)

        logits = self.classifier(x)
        return logits

print("✓ SimpleProteinCNN class created")

sample_model = SimpleProteinCNN(num_classes=len(train_dataset.label_cols))
total_params = sum(p.numel() for p in sample_model.parameters())

print(f"  Model parameters: {total_params:,}")
print_memory_usage("After creating model")

✓ SimpleProteinCNN class created
  Model parameters: 299,305
After creating model - Memory Usage: 0.49 GB


0.4931182861328125

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import gc

device = torch.device("cpu")
print(f"✓ Using device: {device}")

print(f"Training proteins: {len(train_dataset):,}")
print(f"Validation proteins: {len(val_dataset):,}")
print(f"Test proteins: {len(test_dataset):,}")

label_cols = train_dataset.label_cols

if len(label_cols) == 0:
    raise ValueError("No GO term label columns found!")

num_classes = len(label_cols)
print(f"✓ Found {num_classes} GO term labels")

batch_size = 8  # small batch size for CPU

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0
)

print(f"✓ DataLoaders created")
print(f"  Batch size: {batch_size}")
print(f"  Training batches: {len(train_loader):,}")

class LightweightProteinCNN(nn.Module):

    def __init__(self, num_classes):
        super().__init__()

        self.embedding = nn.Embedding(num_embeddings=21, embedding_dim=32, padding_idx=0)
        self.conv = nn.Conv1d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.pool = nn.AdaptiveMaxPool1d(1)
        self.fc = nn.Linear(64, num_classes)
        self.relu = nn.ReLU()

    def forward(self, x):

        if x.dim() == 1:
            x = x.unsqueeze(0)

        x = self.embedding(x)
        x = x.permute(0, 2, 1)
        x = self.relu(self.conv(x))
        x = self.pool(x).squeeze(-1)
        x = self.fc(x)

        return x

model = LightweightProteinCNN(num_classes=num_classes).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"✓ Model created: {model.__class__.__name__}")
print(f"  Total parameters: {total_params:,}")

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=0.0)

print("✓ Loss function and optimizer initialized")

batch = next(iter(train_loader))
sequences = batch["sequence"].to(device)
labels = batch["labels"].to(device)

print(f"Batch shapes:")
print(f"  Sequences: {sequences.shape}")
print(f"  Labels: {labels.shape}")

outputs = model(sequences)
print(f"  Model outputs: {outputs.shape}")

loss = criterion(outputs, labels)
print(f"  Loss: {loss.item():.4f}")

optimizer.zero_grad()
loss.backward()
optimizer.step()

print("✓ Forward and backward pass successful")

gc.collect()

print("FINAL TRAINING SETUP SUMMARY")

print(f"Device: {device}")
print(f"Model: {model.__class__.__name__}")
print(f"Parameters: {total_params:,}")
print(f"Optimizer: {optimizer.__class__.__name__}")
print(f"Batch size: {batch_size}")
print(f"Training samples: {len(train_dataset):,}")
print(f"Validation samples: {len(val_dataset):,}")
print(f"Test samples: {len(test_dataset):,}")

print("\nREADY FOR TRAINING")

✓ Using device: cpu
Training proteins: 71,925
Validation proteins: 15,412
Test proteins: 15,413
✓ Found 2761 GO term labels
✓ DataLoaders created
  Batch size: 8
  Training batches: 8,991
✓ Model created: LightweightProteinCNN
  Total parameters: 186,345
✓ Loss function and optimizer initialized
Batch shapes:
  Sequences: torch.Size([8, 1024])
  Labels: torch.Size([8, 2761])
  Model outputs: torch.Size([8, 2761])
  Loss: 0.7990
✓ Forward and backward pass successful
FINAL TRAINING SETUP SUMMARY
Device: cpu
Model: LightweightProteinCNN
Parameters: 186,345
Optimizer: Adam
Batch size: 8
Training samples: 71,925
Validation samples: 15,412
Test samples: 15,413

READY FOR TRAINING


In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    patience=2,
    factor=0.5
)
num_epochs = 10

checkpoint_path = r"C:\Users\USER\Documents\cod3astro\ML_AI\ProteinSeq_DL\model_prep\scripts\checkpoint.pt"
best_model_path = r"C:\Users\USER\Documents\cod3astro\ML_AI\ProteinSeq_DL\model_prep\models\best_model.pt"

if os.path.exists(checkpoint_path):

    print("✓ Checkpoint found — resuming from previous session...")

    # Load the full checkpoint dictionary from disk.
    checkpoint = torch.load(checkpoint_path, map_location=device)

    # Restore model weights — the learned parameters from last session.
    model.load_state_dict(checkpoint["model_state_dict"])

    # Restore optimizer state — Adam's internal momentum history.
    # Without this, Adam restarts cold and early epochs are wasted.
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

    # Restore scheduler state — its patience counter and current LR.
    scheduler.load_state_dict(checkpoint["scheduler_state_dict"])

    # Restore which epoch we stopped at so the loop continues correctly.
    start_epoch = checkpoint["epoch"] + 1

    # Restore best val loss so saving logic still works correctly.
    best_val_loss = checkpoint["best_val_loss"]

    print(f"  Resuming from epoch {start_epoch + 1}/{num_epochs}")
    print(f"  Best val loss so far: {best_val_loss:.4f}")

else:
    # No checkpoint found — this is a brand new training run.
    print("No checkpoint found — starting fresh...")
    start_epoch = 0
    best_val_loss = float('inf')

# TRAINING LOOP

# range(start_epoch, num_epochs) means:
# - Fresh run:  starts at epoch 0, runs all 10 epochs
# - Resumed run: starts at wherever we left off, skips completed epochs
for epoch in range(start_epoch, num_epochs):

    model.train()
    total_loss = 0

    for batch in train_loader:
        sequences = batch["sequence"].to(device)
        labels = batch["labels"].to(device)
        optimizer.zero_grad()
        outputs = model(sequences)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    train_loss = total_loss / len(train_loader)

    # ── Validation ──
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for batch in val_loader:
            sequences = batch["sequence"].to(device)
            labels = batch["labels"].to(device)
            outputs = model(sequences)
            val_loss += criterion(outputs, labels).item()

    val_loss /= len(val_loader)
    scheduler.step(val_loss)

    # SAVE BEST MODEL (separately, only on improvement)

    if val_loss < best_val_loss:
        best_val_loss = val_loss

        # Save only the model weights to a clean file.
        # This file is what you will use for inference/evaluation later.
        torch.save(model.state_dict(), best_model_path)
        print(f"  ✓ Best model saved (val loss: {val_loss:.4f})")

    # SAVE FULL CHECKPOINT AFTER EVERY EPOCH

    # This saves everything needed to resume training.
    # It overwrites the previous checkpoint — we only ever need the latest one.
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "best_val_loss": best_val_loss,
        "train_loss": train_loss,
        "val_loss": val_loss,
    }, checkpoint_path)

    # ── Epoch Summary ──
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch {epoch+1}/{num_epochs} | Train Loss = {train_loss:.4f} | Val Loss = {val_loss:.4f} | LR = {current_lr:.6f}")
    print(f"  ✓ Checkpoint saved (epoch {epoch+1} complete)")

✓ Checkpoint found — resuming from previous session...
  Resuming from epoch 11/10
  Best val loss so far: 0.0155


In [6]:
# Check one batch of probabilities to see what values the model is actually outputting
with torch.no_grad():
    batch = next(iter(test_loader))
    sequences = batch["sequence"].to(device)
    outputs = model(sequences)
    probs = torch.sigmoid(outputs)

# Print statistics about the probability distribution
print(f"Min probability:    {probs.min().item():.6f}")
print(f"Max probability:    {probs.max().item():.6f}")
print(f"Mean probability:   {probs.mean().item():.6f}")
print(f"Median probability: {probs.median().item():.6f}")

# Show how many predictions would be made at different thresholds
for threshold in [0.5, 0.3, 0.2, 0.1, 0.05, 0.01]:
    count = (probs > threshold).sum().item()
    print(f"  Threshold {threshold} → {count:,} positive predictions")

Min probability:    0.000256
Max probability:    0.227327
Mean probability:   0.002517
Median probability: 0.001197
  Threshold 0.5 → 0 positive predictions
  Threshold 0.3 → 0 positive predictions
  Threshold 0.2 → 13 positive predictions
  Threshold 0.1 → 53 positive predictions
  Threshold 0.05 → 81 positive predictions
  Threshold 0.01 → 593 positive predictions


In [7]:
import torch
import numpy as np
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    classification_report
)

# First collect all probabilities from the full test set
all_probs = []
all_true_labels = []

with torch.no_grad():
    for batch in test_loader:
        sequences = batch["sequence"].to(device)
        labels = batch["labels"].to(device)
        outputs = model(sequences)
        probs = torch.sigmoid(outputs)
        all_probs.append(probs.cpu().numpy())
        all_true_labels.append(labels.cpu().numpy())

all_probs = np.vstack(all_probs)
all_true_labels = np.vstack(all_true_labels)

# Test a range of thresholds and find which gives the best micro F1
print("Threshold Search:")
print("-" * 40)

best_f1 = 0
best_threshold = 0

# Check thresholds from 0.001 to 0.1 in small steps
for threshold in [0.001, 0.002, 0.003, 0.005, 0.007, 0.01, 0.02, 0.03, 0.05, 0.07, 0.1]:
    
    # Apply threshold to get binary predictions
    preds = (all_probs > threshold).astype(int)
    
    # Skip if model predicts nothing or everything
    # as these give misleading F1 scores
    total_preds = preds.sum()
    if total_preds == 0:
        print(f"  Threshold {threshold:.3f} → no predictions, skipping")
        continue
    
    # Calculate micro F1 at this threshold
    f1 = f1_score(all_true_labels, preds, average='micro', zero_division=0)
    precision = precision_score(all_true_labels, preds, average='micro', zero_division=0)
    recall = recall_score(all_true_labels, preds, average='micro', zero_division=0)
    
    print(f"  Threshold {threshold:.3f} → F1: {f1:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f} | Predictions: {total_preds:,}")
    
    # Track the best threshold
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = threshold

print("-" * 40)
print(f"  Best threshold: {best_threshold}")
print(f"  Best Micro F1:  {best_f1:.4f}")

Threshold Search:
----------------------------------------
  Threshold 0.001 → F1: 0.0069 | Precision: 0.0034 | Recall: 0.9228 | Predictions: 28,780,757
  Threshold 0.002 → F1: 0.0102 | Precision: 0.0051 | Recall: 0.7727 | Predictions: 16,217,830
  Threshold 0.003 → F1: 0.0141 | Precision: 0.0071 | Recall: 0.6710 | Predictions: 10,161,556
  Threshold 0.005 → F1: 0.0213 | Precision: 0.0109 | Recall: 0.5491 | Predictions: 5,434,804
  Threshold 0.007 → F1: 0.0279 | Precision: 0.0144 | Recall: 0.4775 | Predictions: 3,576,629
  Threshold 0.010 → F1: 0.0360 | Precision: 0.0188 | Recall: 0.3999 | Predictions: 2,283,936
  Threshold 0.020 → F1: 0.0536 | Precision: 0.0296 | Recall: 0.2870 | Predictions: 1,043,655
  Threshold 0.030 → F1: 0.0691 | Precision: 0.0403 | Recall: 0.2420 | Predictions: 645,833
  Threshold 0.050 → F1: 0.0874 | Precision: 0.0565 | Recall: 0.1930 | Predictions: 367,718
  Threshold 0.070 → F1: 0.0943 | Precision: 0.0653 | Recall: 0.1693 | Predictions: 278,941
  Threshold 0.

In [8]:
# Replace 0.5 with whatever best_threshold was found above
preds = (all_probs > best_threshold).astype(int)

f1_micro   = f1_score(all_true_labels, preds, average='micro',   zero_division=0)
f1_macro   = f1_score(all_true_labels, preds, average='macro',   zero_division=0)
f1_sample  = f1_score(all_true_labels, preds, average='samples', zero_division=0)
precision  = precision_score(all_true_labels, preds, average='micro', zero_division=0)
recall     = recall_score(all_true_labels, preds, average='micro',    zero_division=0)

print("\n" + "="*60)
print("FINAL TEST SET EVALUATION")
print("="*60)
print(f"  Threshold used:    {best_threshold}")
print(f"  Micro F1 Score:    {f1_micro:.4f}  ← most important")
print(f"  Macro F1 Score:    {f1_macro:.4f}")
print(f"  Sample F1 Score:   {f1_sample:.4f}")
print(f"  Precision (micro): {precision:.4f}")
print(f"  Recall (micro):    {recall:.4f}")
print("="*60)


FINAL TEST SET EVALUATION
  Threshold used:    0.1
  Micro F1 Score:    0.1058  ← most important
  Macro F1 Score:    0.0011
  Sample F1 Score:   0.1460
  Precision (micro): 0.0818
  Recall (micro):    0.1498


In [ ]:
# This confirms the exact column names available in your dataset
print("Sample column names:")
print(train_dataset_arrow.column_names[:10])  # print first 10 columns

# We stack them all into one array of shape (71925, 2761)
print("Extracting label columns — this may take a moment...")

label_counts = np.array(
    [train_dataset_arrow[col] for col in label_cols],
    dtype=np.float32
).T.sum(axis=0)  # sum across all proteins for each GO term

# Rare GO terms get a higher weight so the model pays more attention to them.
# Common GO terms get a lower weight since the model already sees them often.
pos_weight = torch.tensor(
    (len(train_dataset_arrow) - label_counts) / (label_counts + 1e-6),
    dtype=torch.float32
).to(device)

# Replace the old criterion with this weighted version
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

print(f"✓ Weighted criterion created")
print(f"  Min weight: {pos_weight.min().item():.2f}")
print(f"  Max weight: {pos_weight.max().item():.2f}")
print(f"  Mean weight: {pos_weight.mean().item():.2f}")

Sample column names:
['sequence', 'accession', 'label_GO:0005737', 'label_GO:0005634', 'label_GO:0005829', 'label_GO:0005886', 'label_GO:0005576', 'label_GO:0046872', 'label_GO:0005524', 'label_GO:0016020']
Extracting label columns — this may take a moment...
